# LLM-SR — Scenario 3 E-LLM-SR

**Set `SCENARIO` in the scenario cell** to `"S1"`, `"S2"`, `"S3"`, or `"S4"` and run — everything else is identical across the four (engine, Boltzmann sampling, generalisation-aware selection, neutral seed).

- **S1** none
- **S2** competing baselines: risk-neutral, linear, anticipated-regret
- **S3** the SCP distortion model
- **S4** all of S2 + S3

**Selection (uniform TRAIN only; test never touched):** `cv_mse + DSHIFT_W*dist_breakdown + RESP_W*responsiveness + COMPLEXITY_PENALTY*n_params`. Run RA then AB; results vs SCP appear at the end.

## 1. Imports, API client, cache, cost counter

In [17]:
import os, re, json, textwrap, warnings, hashlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar
from scipy.integrate import quad
from scipy import stats
import anthropic
warnings.filterwarnings("ignore")

# ---- reproducibility ----
GLOBAL_SEED = 20240620
np.random.seed(GLOBAL_SEED)

# ---- API KEY ----
API_KEY = ""   # <-- paste

_key = API_KEY or os.environ.get("ANTHROPIC_API_KEY", "")
if not _key:
    print("WARNING: no API key. Paste one into API_KEY above or export ANTHROPIC_API_KEY; "
          "the deterministic cells (data, helpers, SCP) still run without it.")
try:
    client = anthropic.Anthropic(api_key=_key) if _key else None
except Exception as _e:
    client = None
    print("No API client yet:", _e)

MODEL  = "claude-sonnet-4-6"

# pricing (USD per 1M tokens) for the live cost counter; auto-selected from MODEL
_PRICES = {"claude-opus-4-8": (15.0, 75.0), "claude-opus-4-7": (15.0, 75.0),
           "claude-sonnet-4-6": (3.0, 15.0)}
_pi, _po = _PRICES.get(MODEL, (15.0, 75.0))
PRICE_IN, PRICE_OUT = _pi / 1_000_000, _po / 1_000_000
# Opus 4.8/4.7 and Fable 5 reject `temperature` (400); Sonnet 4.6 accepts it.
MODEL_ACCEPTS_TEMPERATURE = not any(t in MODEL for t in ("opus-4-8", "opus-4-7", "fable-5"))
SPEND = {"in_tok": 0, "out_tok": 0, "usd": 0.0, "calls": 0, "cached": 0}

# ---- on-disk prompt cache (identical prompts are free on re-run) ----
CACHE_PATH = "llm_cache_SCP_consistsimpl.json"
_cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
def _save_cache(): json.dump(_cache, open(CACHE_PATH, "w"))

def call_llm(prompt, temperature=0.8, nonce="", use_cache=False):
    if client is None:
        raise RuntimeError("No API client: set ANTHROPIC_API_KEY in the environment.")
    key = hashlib.sha256((MODEL+str(temperature)+str(nonce)+prompt).encode()).hexdigest()
    if use_cache and key in _cache:
        SPEND["cached"] += 1
        return _cache[key]
    # Opus 4.8/4.7 and Fable 5 reject `temperature` (400); Sonnet 4.6 accepts it. Pass it only when
    # the backbone allows, so Sonnet keeps the LLM-SR sampling-diversity knob and Opus doesn't 400.
    _kw = {"temperature": temperature} if MODEL_ACCEPTS_TEMPERATURE else {}
    resp = client.messages.create(model=MODEL, max_tokens=1500,
                                  timeout=90,   # cap any single request
                                  messages=[{"role": "user", "content": prompt}], **_kw)
    u = resp.usage
    SPEND["in_tok"]  += u.input_tokens
    SPEND["out_tok"] += u.output_tokens
    SPEND["usd"]     += u.input_tokens*PRICE_IN + u.output_tokens*PRICE_OUT
    SPEND["calls"]   += 1
    txt = resp.content[0].text
    if use_cache: _cache[key] = txt; _save_cache()
    return txt

print("Model:", MODEL)
print("API key configured:", bool(_key))


Model: claude-sonnet-4-6
API key configured: True


In [18]:
# ---- API SMOKE TEST: run once to confirm the backbone works BEFORE the search ----
# Lists the models your key can access, then makes one real call. This surfaces the TRUE
# error (e.g. model not enabled on your key) instead of letting the search loop swallow it.
print("Models available to this API key:")
try:
    for _m in client.models.list():
        print("   ", _m.id)
except Exception as _e:
    print("   could not list models:", type(_e).__name__, "-", str(_e)[:200])

print(f"\nTest call with MODEL = {MODEL} ...")
try:
    _r = client.messages.create(model=MODEL, max_tokens=64,
                                messages=[{"role": "user", "content": "reply with OK"}])
    print("   SUCCESS:", _r.content[0].text.strip()[:80])
    print("   usage:", _r.usage.input_tokens, "in /", _r.usage.output_tokens, "out")
except Exception as _e:
    print("   FAILED:", type(_e).__name__, "-", str(_e)[:300])
    print("   -> NotFoundError / 404  : MODEL not enabled on your key. Set MODEL (cell 2) to one listed above")
    print("                             (e.g. 'claude-opus-4-7' or 'claude-sonnet-4-6') and re-run from cell 2.")
    print("   -> AuthenticationError / 401 : the API key is wrong.")
    print("   -> PermissionDeniedError / 403: key lacks access to this model/feature.")


Models available to this API key:
    claude-sonnet-5
    claude-fable-5
    claude-opus-4-8
    claude-opus-4-7
    claude-sonnet-4-6
    claude-opus-4-6
    claude-opus-4-5-20251101
    claude-haiku-4-5-20251001
    claude-sonnet-4-5-20250929
    claude-opus-4-1-20250805

Test call with MODEL = claude-sonnet-4-6 ...
   SUCCESS: OK
   usage: 10 in / 4 out


## 2. Load data
RA target = `reserveprice`. AB target = `offer`, **rejected rows only** (the ex-post reserve price only exists when the auctioneer rejects the auction price and counter-offers).

In [19]:
def _load(path, sep, cols):
    d = pd.read_csv(path, sep=sep)
    for c in cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    return d

# We KEEP the auctioneer id `name` (64 auctioneers) so the final paired tests can cluster by it.
RA_COLS = ["value","bidN","Period","reserveprice","name"]
ra_train = _load("RA with auction price.csv", ";", ["value","bidN","Period","reserveprice"]) \
              [RA_COLS].dropna().reset_index(drop=True)
ra_test  = _load("RA-4p.csv", ",", ["value","bidN","Period","reserveprice"]) \
              [RA_COLS].dropna().reset_index(drop=True)

# AB (ex-post): BARGAINED rows only (agree == -1) in BOTH train and test.
AB_COLS = ["value","bidN","Period","auctionprice","offer","name"]
_abtr = _load("AB with auction price.csv", ";", ["value","bidN","Period","auctionprice","offer","agree"])
_abte = _load("AB-4p.csv", ",", ["value","bidN","Period","auctionprice","offer","agree"])
ab_train = _abtr[_abtr["agree"] == -1][AB_COLS].dropna().reset_index(drop=True)
ab_test  = _abte[_abte["agree"] == -1][AB_COLS].dropna().reset_index(drop=True)

print(f"RA  uniform train {len(ra_train):,} | skewed test {len(ra_test):,} | auctioneers {ra_test['name'].nunique()}")
print(f"AB  uniform train {len(ab_train):,} (bargained) | skewed test {len(ab_test):,} | auctioneers {ab_test['name'].nunique()}")


RA  uniform train 5,760 | skewed test 5,760 | auctioneers 64
AB  uniform train 4,581 (bargained) | skewed test 4,420 | auctioneers 64


## 3. Cost distribution helper + metrics
`F(c)` is the cost CDF. **This is the only distribution information the LLM gets**, and it is swapped from power=1 (train) to power=4 (test) automatically. Everything else the LLM must discover.

In [20]:
C_MAX = 100.0
POWER_TRAIN, POWER_TEST = 1, 4

def F_cost(c, power): return np.clip(c/C_MAX, 0, 1)**power
def f_cost(c, power):
    c = np.clip(c, 1e-9, C_MAX); return power*c**(power-1)/C_MAX**power

def mse(p, a): return float(np.mean((np.asarray(p)-np.asarray(a))**2))
def mse_by_n(p, a, b):
    p, a, b = map(np.asarray, (p, a, b))
    o = {n: float(np.mean((p[b==n]-a[b==n])**2)) for n in [1,2,3]}
    o["pooled"] = mse(p, a); return o

# helpers handed to every discovered equation. F/f carry the correct power automatically.
def make_helpers(power):
    # F/f: cost CDF/PDF over [0,100] (used by RA's competition conditional probability)
    # D: ex-post conditional prob that winner accepts offer r given clearing price c2 = (r/c2)^power
    #    (this is the AB-specific conditional probability; r and c2 are the offer and auction price)
    def D(r, c2):
        r = np.asarray(r, dtype=float); c2 = np.asarray(c2, dtype=float)
        ratio = np.clip(np.where(c2>0, r/np.maximum(c2,1e-9), 0.0), 0.0, 1.0)
        return ratio**power
    def G(r, n):
        # RA competition conditional probability, built from F so it carries the cost distribution
        n = np.asarray(n, dtype=float)
        Fr = F_cost(r, power)
        return np.clip(n * Fr / ((n - 1) * Fr + 1), 1e-9, 1 - 1e-9)
    return {"F": (lambda c: F_cost(c, power)),
            "f": (lambda c: f_cost(c, power)),
            "D": D, "G": G,
            "power": power, "C_MAX": C_MAX, "np": np}

print("Helper check (uniform vs skewed):")
print("  F(50) power=1:", round(F_cost(50,1),4), "| power=4:", round(F_cost(50,4),4))

Helper check (uniform vs skewed):
  F(50) power=1: 0.5 | power=4: 0.0625


## 4. SCP benchmark — computed but kept SEPARATE (revealed only at the end)
These functions produce the SCP predictions used purely as the external yardstick. **They are never shown to the LLM and never enter selection.** Parameters are the paper's MLE estimates.

In [21]:
MU_EA, LAM_EA = 1.57, 1.42   # RA (paper Table 3, Model 9)
MU_EP, LAM_EP = 1.45, 1.25   # AB (paper Table 3, Model 7)

def psi(K, mu, lam):
    K = np.clip(K, 1e-12, 1-1e-12); return np.exp(-mu*(-np.log(K))**lam)

def scp_ea(v, n, mu, lam, power):
    def nu(r):
        if r <= 0 or r >= v: return 0.0
        Fr = F_cost(r, power); G = n*Fr/((n-1)*Fr+1); TG = psi(G, mu, lam)
        J  = n*Fr*(1-Fr)**(n-1) + (1-Fr)**n
        integ, _ = quad(lambda y: (v-y)*n*(n-1)*F_cost(y,power)*f_cost(y,power)*(1-F_cost(y,power))**(n-2), 0, r, limit=50)
        return -((v-r)*TG*J + integ)
    return float(np.clip(minimize_scalar(nu, bounds=(1e-6, min(v,99.99)), method="bounded").x, 0, v))

def scp_ep(v, c2, mu, lam, power):
    if c2 <= 0: return 0.0
    def nu(rp): K = np.clip((rp/c2)**power, 1e-12, 1-1e-12); return -((v-rp)*psi(K, mu, lam))
    r = float(np.clip(minimize_scalar(nu, bounds=(1e-6, c2-1e-6), method="bounded").x, 0, c2))
    ub = (v-r)*psi(np.clip((r/c2)**power,1e-12,1-1e-12), mu, lam)
    return c2 if (v-c2) >= ub else r

print("Computing SCP benchmark on TEST sets (left-skewed, power=4) ...")
ra_test["_scp"] = [scp_ea(r.value, r.bidN, MU_EA, LAM_EA, POWER_TEST) for _, r in ra_test.iterrows()]
ab_test["_scp"] = [scp_ep(r.value, r.auctionprice, MU_EP, LAM_EP, POWER_TEST) for _, r in ab_test.iterrows()]
ab_test["_scp"] = np.minimum(ab_test["_scp"], ab_test["auctionprice"])

SCP_RA = mse_by_n(ra_test["_scp"], ra_test["reserveprice"], ra_test["bidN"])
SCP_AB = mse_by_n(ab_test["_scp"], ab_test["offer"],        ab_test["bidN"])
print(f"SCP RA test pooled {SCP_RA['pooled']:.2f}  (paper 73.50)  byn {SCP_RA[1]:.1f}/{SCP_RA[2]:.1f}/{SCP_RA[3]:.1f}")
print(f"SCP AB test pooled {SCP_AB['pooled']:.2f}  (paper 65.22)  byn {SCP_AB[1]:.1f}/{SCP_AB[2]:.1f}/{SCP_AB[3]:.1f}")

Computing SCP benchmark on TEST sets (left-skewed, power=4) ...
SCP RA test pooled 73.50  (paper 73.50)  byn 49.5/81.5/89.5
SCP AB test pooled 64.10  (paper 65.22)  byn 78.2/62.5/48.1


## 5. Theory-grounded prompts (B1) — separate for RA and AB
The two formats describe **different phenomena**: RA is *anticipatory* (uncertainty about unseen competition); AB is *reactive* (uncertainty about whether the winner accepts a counter-offer below the observed price). Both describe the S-shaped conditional-probability distortion **in words only** — no ψ formula, no G(r), no integral.

Seeds are minimal (risk-neutral baseline), NOT SCP — so any SCP-like structure is genuinely discovered.

In [22]:
# ===== ONE KNOB: choose the scenario. Everything else is identical across S1-S4. =====
SCENARIO = "S3"     # "S1" (none) | "S2" (linear/RN/regret) | "S3" (SCP) | "S4" (all)

# ----- NEUTRAL seed: identical for every scenario; contains NO theory and NO answer. -----
NEUTRAL_SEED_RA = textwrap.dedent("""
def equation(X, params, helpers):
    value = X[:, 0]; bidN = X[:, 1]; Period = X[:, 2]
    np_ = helpers["np"]
    # neutral starting point (no theory) -- the search builds structure from the PRIOR KNOWLEDGE in the prompt
    return np_.clip(params[0] * value, 0, 100)
""")
NEUTRAL_SEED_AB = textwrap.dedent("""
def equation(X, params, helpers):
    value = X[:, 0]; bidN = X[:, 1]; Period = X[:, 2]; auctionprice = X[:, 3]
    np_ = helpers["np"]
    # neutral starting point (no theory)
    return np_.clip(params[0] * auctionprice, 0, auctionprice)
""")

# ----- PRIOR_KNOWLEDGE: the ONLY thing that differs across scenarios (the experimental variable). -----
PK_S3_RA = textwrap.dedent("""
PRIOR KNOWLEDGE -- the Subjective Conditional Probability (SCP) model and its extensions for the EX-ANTE
reserve price (Shachat & Tan, 2023).
CORE MECHANISM (defining SCP): the auctioneer chooses the reserve r that maximises a SUBJECTIVELY DISTORTED
surplus. Use a VECTORISED GRID SEARCH (argmax -- a HARD maximiser, not a temperature-weighted average) over
candidate reserves r of
    (value - r) * psi(G(r))
  - G(r) = helpers["G"](r, n)  is the conditional probability the reserve sets the price, already built from
    the cost distribution -- USE IT DIRECTLY, do NOT rebuild it from F.
  - psi(K) = exp(-mu * (-ln K)**lambda)  is the S-shaped distortion (mu>0, lambda>1); lambda>1 makes the
    reserve DECREASE in the number of bidders.
FULL EX-ANTE MODEL (exact payoff, optional -- matches SCP in-sample but does not by itself beat it):
    (value-r)*psi(G(r))*J(r) + integral_0^r (value-y)*f2(y) dy
  with J(r)=n*F(r)*(1-F(r))**(n-1)+(1-F(r))**n the binding probability, and f2(y)=n(n-1)F(y)**(n-2)f(y)(1-F(y))
  (f=helpers[\"f\"]) the UNDISTORTED competition surplus. If you use the integral, compute it as a VECTORISED
  np.cumsum over the SAME grid -- NEVER scipy.quad, NEVER a per-row loop, NEVER an inner optimiser (speed check).
EXTENSIONS S&T FOUND IMPROVE THE FIT (you may discover and use these):
  - BIDDER-DEPENDENT DISTORTION (their Models 8/10): let the distortion depend on the number of bidders, e.g.
    mu(n) = mu0 + mu1*(n-2). This captures how reserves fall further as bidders increase, and tends to improve
    BOTH in-sample and out-of-sample fit.
  - LEARNING: SCP ignores experience, but reserves drift over the 90 periods; a bounded trend
    (1 + theta*(Period/90 - 0.5)) helps.
This is your PRIOR. You MAY use it as-is, SIMPLIFY, EXTEND, or REPLACE it -- the goal is the equation that
GENERALISES best to the unknown cost distribution, not the one that recites theory. Keep ALL cost-distribution
dependence inside helpers[\"F\"]/[\"f\"]. Do NOT add affine terms in value.
STABILITY: lam = 1.0 + 2.0/(1.0+np.exp(-params[k])); for the distortion level use a bounded base
mu0 = 0.5 + 2.5/(1.0+np.exp(-params[j])) and (optionally) a small slope mu1 = 0.5*np.tanh(params[m]).
""")

PK_S3_AB = textwrap.dedent("""
PRIOR KNOWLEDGE -- the Subjective Conditional Probability (SCP) model for the EX-POST counter-offer
(Shachat & Tan, 2023).
CORE MECHANISM: after observing the auction price c2, choose the offer r by a VECTORISED GRID SEARCH (argmax --
a HARD maximiser) over candidate offers of the COMPLETE ex-post objective
    (value - r) * psi(D(r, c2))
  - D(r, c2) = helpers[\"D\"](r, c2)  is the conditional probability the winner accepts the offer r given c2.
  - psi(K) = exp(-mu * (-ln K)**lambda)  is the S-shaped distortion; lambda>1 makes the offer INCREASE with c2.
There is NO binding-probability weight and NO competition integral here: once c2 is observed the auction
uncertainty is RESOLVED, so this objective is already COMPLETE (unlike the ex-ante case).
EXTENSIONS (you may discover and use): BIDDER-DEPENDENT distortion mu(n)=mu0+mu1*(n-2) (S&T Models 8/10);
a bounded LEARNING trend (1 + theta*(Period/90 - 0.5)).
This is your PRIOR. You MAY use it as-is, EXTEND, or REPLACE it -- goal = best generalisation. Keep distribution
dependence inside helpers[\"D\"]. Do NOT add affine terms in value.
STABILITY: lam = 1.0 + 2.0/(1.0+np.exp(-params[k])); mu0 = 0.5 + 2.5/(1.0+np.exp(-params[j])),
optional slope mu1 = 0.5*np.tanh(params[m]).
""")

PK_S1 = "PRIOR KNOWLEDGE -- none. No theory is provided; discover the structure entirely from the data."
PK_S2_RA = textwrap.dedent("""
PRIOR KNOWLEDGE -- competing behavioural models for the EX-ANTE reserve price (Shachat & Tan, 2023), WITHOUT
the SCP distortion. You may use, combine, or improve any of them.
(1) RISK-NEUTRAL benchmark: a rational auctioneer maximises EXPECTED surplus with the TRUE (undistorted)
    probability. Use a VECTORISED GRID SEARCH (argmax) over candidate reserves r of
        (value - r) * G(r),   G(r) = helpers["G"](r, n)
    where G(r) is the conditional probability the reserve sets the price. This benchmark is roughly INVARIANT
    to the number of bidders.
(2) LINEAR reduced form: the reserve as a linear function of value, the number of bidders, a value*bidder
    interaction, and the period.
(3) ANTICIPATED REGRET (Davis et al., 2011): the auctioneer trades off the regret of a reserve set too HIGH
    (overpaying) against one too LOW (failing to buy); anticipating this makes the reserve DECREASE in the
    number of bidders.
Empirically, human reserves DECREASE in the number of bidders. Keep ALL cost-distribution dependence inside
helpers["F"]/["f"] so the equation transfers across distributions. Do NOT add affine terms in value. Prefer
FEW parameters (2-3); terms that only help the training distribution will not transfer.
""")

PK_S2_AB = textwrap.dedent("""
PRIOR KNOWLEDGE -- competing behavioural models for the EX-POST counter-offer (Shachat & Tan, 2023), WITHOUT
the SCP distortion. You may use, combine, or improve any of them.
(1) RISK-NEUTRAL benchmark: after observing the auction price c2, a rational auctioneer chooses the counter-
    offer r (below c2) maximising EXPECTED surplus with the TRUE probability. Use a VECTORISED GRID SEARCH
    (argmax) over candidate offers of
        (value - r) * D(r, c2),   D(r, c2) = helpers["D"](r, c2)
    where D is the probability the winning supplier accepts the offer r given c2.
(2) LINEAR reduced form: the counter-offer as a linear function of value, the number of bidders, the observed
    auction price, and the period.
Empirically, human counter-offers INCREASE with the observed auction price. Keep distribution dependence inside
helpers["D"]. Do NOT add affine terms in value. Prefer FEW parameters (2-3).
""")

# S4 = ALL priors: the competing baselines (S2) AND the SCP model + extensions (S3), concatenated.
PK_S4_RA = PK_S2_RA + PK_S3_RA
PK_S4_AB = PK_S2_AB + PK_S3_AB

PRIOR_KNOWLEDGE = {
  "S1": {"RA": PK_S1,        "AB": PK_S1},
  "S2": {"RA": PK_S2_RA,     "AB": PK_S2_AB},
  "S3": {"RA": PK_S3_RA,     "AB": PK_S3_AB},
  "S4": {"RA": PK_S4_RA,     "AB": PK_S4_AB},
}

# D = F(r)/F(c2) is assembled model structure (used by the rational and SCP models), not raw environment.
# Exposed whenever a model is provided (S2/S3/S4); withheld only in the no-theory baseline S1, where the
# search must build it from F -- just as RA-S1 must build G(r) from F.
EXPOSE_D = SCENARIO != "S1"

TASKS = {
  "RA": dict(
    fmt="ex-ante reserve-price", theory=PRIOR_KNOWLEDGE[SCENARIO]["RA"], seed=NEUTRAL_SEED_RA, helper_extra=('  helpers["G"](r, n) = conditional probability the reserve r sets the price given n bidders,\n'
                  '                     built from the cost distribution (carries the cost distribution).\n') if EXPOSE_D else "",
    pred_cols=["value","bidN","Period"], target="reserveprice",
    train="ra_train", test="ra_test", scp=SCP_RA, benchmark=73.50,
    io=("INPUTS  X[:,0]=value (auctioneer valuation, 50-150)  X[:,1]=bidN (number of bidders: 1,2,3)\n"
        "        X[:,2]=Period (auction index 1-90)\n"
        "OUTPUT  reserveprice in [0,100]"),
    bound="clip(result, 0, 100)"),
  "AB": dict(
    fmt="ex-post counter-offer", theory=PRIOR_KNOWLEDGE[SCENARIO]["AB"], seed=NEUTRAL_SEED_AB,
    helper_extra=('  helpers["D"](r, c2) = conditional probability the winning supplier accepts a counter-offer r,\n'
                  '                     given the observed auction price c2 (carries the cost distribution).\n') if EXPOSE_D else "",
    pred_cols=["value","bidN","Period","auctionprice"], target="offer",
    train="ab_train", test="ab_test", scp=SCP_AB, benchmark=64.10,
    io=("INPUTS  X[:,0]=value  X[:,1]=bidN  X[:,2]=Period  X[:,3]=auctionprice (observed clearing price)\n"
        "OUTPUT  offer (counter-offer) in [0, auctionprice]"),
    bound="clip(result, 0, auctionprice)"),
}
print(f"SCENARIO={SCENARIO} | EXPOSE_D={EXPOSE_D} | neutral seed, theory only in prompt. Tasks: {list(TASKS)}")


SCENARIO=S3 | EXPOSE_D=True | neutral seed, theory only in prompt. Tasks: ['RA', 'AB']


In [23]:
def build_prompt(task, experience):
    cfg = TASKS[task]
    d_helper = '/["D"]' if task == "AB" else ""
    return textwrap.dedent(f"""
    You are discovering a mathematical equation that predicts how HUMAN auctioneers behave in an
    {cfg['fmt']} English procurement auction. Reason briefly, then return ONE python function.

    IMPORTANT -- YOU ARE EVALUATED ON A DIFFERENT, UNKNOWN COST DISTRIBUTION:
    Your equation is FITTED on data drawn from one supplier-cost distribution but SCORED, without refitting,
    on data from a DIFFERENT cost distribution of UNKNOWN shape. You never see the scoring distribution.
    Therefore EVERY dependence on the cost distribution MUST be expressed through the provided helpers
    helpers["F"]/["f"]{d_helper} -- do NOT assume the distribution is uniform, do NOT hard-code shape
    constants such as (c/100) or (c/100)**k, and do NOT bake in any number that only holds for the fitting
    distribution. Equations that route their distribution-dependence through the helpers transfer across cost
    environments; equations that implicitly encode the fitting shape will fail when scored.

    {cfg['io']}

    HELPERS available inside your function:
      helpers["F"](c)  = the cost CDF F(c) for supplier costs in [0, 100]. Its SHAPE DIFFERS between the data
                         you are fitted on and the data you are scored on -- always call it instead of writing
                         the shape yourself.
      helpers["f"](c)  = the matching density f(c).
    {cfg['helper_extra']}  helpers["np"]    = numpy.

    {cfg['theory']}

    RULES
    - Return exactly one function: def equation(X, params, helpers): ... return <array of predictions>.
    - params is a numpy array of free numeric constants you introduce; they are fitted to the data for you.
      Use AT MOST {MAX_PARAMS} parameters (equations with more are REJECTED). Spend them on the S-shape
      (mu, lambda) and AT MOST ONE small distribution-general adjustment (a bounded Period/learning
      trend). Do NOT add separate affine terms in value (no extra alpha*value or additive
      value-scaled shifts); those overfit the fitting distribution and do not transfer.
    - Bound the output: {cfg['bound']}.
    - VECTORISE (REQUIRED): compute the result with numpy operations over ALL rows of X at once. You MAY
      implement the SCP argmax with a SMALL vectorised grid search (e.g. ~40-60 candidate reserves/offers
      broadcast over the rows, then argmax) -- this is one acceptable way to choose the optimal value. But
      do NOT use Python loops over rows, and do NOT call optimisers (minimize / minimize_scalar) or
      root-finders inside the function -- those are far too slow and are REJECTED by an automatic speed check.
    - Output a brief reasoning comment, then a single ```python code block.

    CURRENT BEST EQUATIONS (lower selection score = better GENERALISATION, not merely lower fitting MSE):
    {experience}

    SEED (a starting point implementing the structure above -- improve its generalisation):
    ```python
    {cfg['seed']}
    ```
    """)
print("Anti-anchoring prompt builder ready.")


Anti-anchoring prompt builder ready.


## 6. Evaluation engine
Compile the proposed equation, fit its parameters by **least squares on the uniform train data** (BFGS / Nelder-Mead with restarts), and record train MSE, parsimony-penalised selection score, and (for observation only) test MSE under the left-skewed distribution.

**Convergence config** matches your handoff: `patience=100`, `min_delta=0.01`, `maxiter=500`, `ftol=1e-9`. `max_iterations=500` by default to cap cost.

In [24]:
import signal
class _Timeout(Exception): pass
def _alarm(signum, frame): raise _Timeout()

SAFE = {"np": np}
COMPLEXITY_PENALTY = 2.0     # added to the score per parameter (parsimony tie-breaker, SCP-blind)
MAX_PARAMS = 4               # SCP core (scale, mu, lambda) + AT MOST one period/learning term
FIT_RESTARTS = 20            # higher: grid-argmax loss is piecewise-flat, so fitting is a random search
FIT_MAXITER  = 500
FIT_FTOL     = 1e-9

# ---- robustness-based selection (computed ONLY on the uniform TRAIN set; test never touched) ----
N_FOLDS         = 5          # k-fold cross-validation for the generalisation estimate
FIT_RESTARTS_CV = 4          # restarts per fold refit
EVAL_TIMEOUT   = 60          # seconds per candidate (bootstrap adds refits)

# ---- Option A: distribution-shift robustness penalty (probe synthetic cost shapes; TEST never used) ----
PROBE_POWERS = [2.0, 3.0]    # synthetic cost-distribution shifts, distinct from the test (POWER_TEST = 4)
DSHIFT_W     = 50.0          # weight on distribution-shift breakdown (tune during the pilot)

# ---- responsiveness requirement: a generaliser MUST respond when the cost distribution changes ----
RESP_W     = 1000.0          # weight; large so distribution-IGNORING equations are effectively rejected
RESP_FLOOR = 1.0             # min mean prediction movement (output units) across the probe shifts

def detect_n_params(s):
    idx = [int(x) for x in re.findall(r"params\[(\d+)\]", s)]
    return (max(idx)+1) if idx else 1

def extract_code(t):
    for pat in [r"```python\n(.*?)```", r"```\n(.*?)```"]:
        m = re.search(pat, t, re.DOTALL)
        if m: return m.group(1).strip()
    m = re.search(r"(def equation\(.*)", t, re.DOTALL)
    return m.group(1).strip() if m else t.strip()

def compile_equation(s):
    ns = dict(SAFE); exec(s, ns)
    fn = ns.get("equation")
    if fn is None: raise ValueError("no equation()")
    return fn

def fit_params(fn, X, y, power, n_params, restarts=FIT_RESTARTS, seed0=0):
    h = make_helpers(power); best_p, best_l = None, np.inf
    for s in range(restarts):
        p0 = np.random.default_rng(GLOBAL_SEED + 1000*seed0 + s).standard_normal(n_params) * 0.5
        try:
            r = minimize(lambda p: np.mean((np.asarray(fn(X, p, h)) - y)**2), p0,
                         method="L-BFGS-B", options={"maxiter": FIT_MAXITER, "ftol": FIT_FTOL})
            if np.isfinite(r.fun) and r.fun < best_l:
                best_l, best_p = float(r.fun), r.x
        except Exception:
            continue
    return best_p, best_l

def cv_mse_score(fn, X, y, n_params):
    """5-fold cross-validated MSE on the uniform train set (the generalisation estimate).
       Fit on 4 folds, score on the held-out fold; average over folds. The TEST distribution is
       never touched. Lower is better; returns None if it cannot be estimated."""
    N = len(y); h = make_helpers(POWER_TRAIN)
    idx = np.random.default_rng(GLOBAL_SEED + 77).permutation(N)
    folds = np.array_split(idx, N_FOLDS)
    errs = []
    for k in range(N_FOLDS):
        te = folds[k]
        tr = np.concatenate([folds[j] for j in range(N_FOLDS) if j != k])
        p, _ = fit_params(fn, X[tr], y[tr], POWER_TRAIN, n_params,
                          restarts=FIT_RESTARTS_CV, seed0=k+1)
        if p is None: return None
        try:
            pred = np.asarray(fn(X[te], p, h), dtype=float)
        except Exception:
            return None
        if not np.all(np.isfinite(pred)): return None
        errs.append(mse(pred, y[te]))
    return float(np.mean(errs)) if errs else None

def dist_shift_penalty(fn, X, params):
    """Penalise candidates that BREAK DOWN under synthetic cost distributions (probe powers).
       Uses ONLY the train inputs and synthetic F/f/D; the test distribution is never touched.
       0 = well-behaved across distributions; larger = collapses/explodes when the distribution changes."""
    try:
        base = np.asarray(fn(X, params, make_helpers(POWER_TRAIN)), dtype=float)
    except Exception:
        return 1.0
    sbase = float(np.nanstd(base)) + 1e-9
    pen = 0.0
    for pw in PROBE_POWERS:
        try:
            pp = np.asarray(fn(X, params, make_helpers(pw)), dtype=float)
        except Exception:
            pen += 1.0; continue                       # crashes under a shifted distribution
        nan_frac = float(np.mean(~np.isfinite(pp)))    # NaN/Inf survive the clip -> detectable
        fin = pp[np.isfinite(pp)]
        spread = (float(np.std(fin)) / sbase) if len(fin) else 0.0
        blow = 1.0 if (spread < 0.1 or spread > 10.0) else 0.0   # predictions collapse or explode
        pen += nan_frac + 0.5 * blow
    return pen / len(PROBE_POWERS)

def responsiveness(fn, X, params):
    """How much do predictions MOVE when the cost distribution changes? A generalisable equation
       routes through F/f/D, so its predictions shift under a probe distribution; a distribution-
       IGNORING equation (e.g. a plain function of value) gives identical predictions -> ~0 here.
       Uses ONLY the train inputs and synthetic F/f/D; the test distribution is never touched."""
    try:
        base = np.asarray(fn(X, params, make_helpers(POWER_TRAIN)), dtype=float)
    except Exception:
        return 0.0
    moves = []
    for pw in PROBE_POWERS:
        try:
            pp = np.asarray(fn(X, params, make_helpers(pw)), dtype=float)
        except Exception:
            continue
        d = np.abs(pp - base); d = d[np.isfinite(d)]
        if len(d): moves.append(float(np.mean(d)))
    return float(np.mean(moves)) if moves else 0.0

def evaluate(code_str, Xtr, ytr, Xte, yte, bte):
    try: fn = compile_equation(code_str)
    except Exception: return None
    npar = detect_n_params(code_str)
    if npar > MAX_PARAMS: return None                      # parameter cap
    # sanity + SPEED PROBE: an equation slow on a small batch will be ~46x slower across all refits.
    # This rejects per-row loops / inner optimisers / huge grids BEFORE they can stall the search
    # (the SIGALRM timeout below cannot interrupt a single long-running numpy/scipy C call).
    try:
        _probe = Xtr[:min(256, len(Xtr))]
        _t0 = time.time()
        _ = fn(_probe, np.ones(max(npar, 1)), make_helpers(POWER_TRAIN))
        if (time.time() - _t0) > 0.25:        # too slow on 256 rows -> reject as non-vectorised
            return None
    except Exception:
        return None
    try:
        signal.signal(signal.SIGALRM, _alarm); signal.alarm(EVAL_TIMEOUT)
        p, tr = fit_params(fn, Xtr, ytr, POWER_TRAIN, npar)
        if p is None or not np.isfinite(tr):
            signal.alarm(0); return None
        cv = cv_mse_score(fn, Xtr, ytr, npar)
    except _Timeout:
        signal.alarm(0); return None
    finally:
        signal.alarm(0)
    if cv is None: return None
    # OBSERVATION-ONLY out-of-sample evaluation (never used for selection)
    try:
        tp = np.asarray(fn(Xte, p, make_helpers(POWER_TEST)))
        if not np.all(np.isfinite(tp)): return None
        tall = mse_by_n(tp, yte, bte)
    except Exception:
        return None
    dist_pen = dist_shift_penalty(fn, Xtr, p)
    resp     = responsiveness(fn, Xtr, p)
    resp_pen = RESP_W * max(0.0, RESP_FLOOR - resp)        # penalise equations that IGNORE the distribution
    select_score = cv + DSHIFT_W*dist_pen + resp_pen + COMPLEXITY_PENALTY*npar
    return dict(code=code_str, params=p, n_params=npar,
                train_mse=tr, cv_mse=cv, dist_pen=dist_pen, resp=resp,
                test_mse=tall["pooled"], test_by_n=tall,
                select_score=select_score)
print("Engine ready (selection = CV-MSE + DSHIFT_W*dist_breakdown + RESP_W*responsiveness_penalty + parsimony).")


Engine ready (selection = CV-MSE + DSHIFT_W*dist_breakdown + RESP_W*responsiveness_penalty + parsimony).


## 7. LLM-SR search loop
Islands + experience buffer (paper-faithful). Selection and the patience counter use the **SCP-blind** `select_score` (train MSE + parsimony). Test MSE is logged only. The loop prints a **live cost estimate** every 25 iterations and respects the convergence rule.

In [25]:
def get_arrays(task):
    cfg = TASKS[task]; dtr = globals()[cfg["train"]]; dte = globals()[cfg["test"]]
    cols, tgt = cfg["pred_cols"], cfg["target"]
    return (dtr[cols].values, dtr[tgt].values,
            dte[cols].values, dte[tgt].values, dte["bidN"].values)

# Boltzmann (FunSearch/LLM-SR) sampling of in-context examples.
# TAU_SCORE: temperature on select_score (MSE units). Smaller -> greedier; larger -> more diverse.
# TAU_LEN:   temperature on (normalised) program length. Favours SHORTER programs (parsimony bias).
TAU_SCORE = 20.0
TAU_LEN   = 1.0

def fmt_exp(island, k=3):
    progs = [b for b in island if b]
    if not progs: return "(none yet -- improve on the seed)"
    if len(progs) <= k:
        chosen = sorted(progs, key=lambda d: d["select_score"])
    else:
        sel  = np.array([d["select_score"] for d in progs], float)
        lens = np.array([len(d["code"])     for d in progs], float)
        s = sel - sel.min()                                   # >=0, 0 for the island's best
        l = (lens - lens.min()) / (lens.max() - lens.min() + 1e-9)   # 0..1, 0 for shortest
        logw = -(s / TAU_SCORE) - (l / TAU_LEN)               # higher quality & shorter => higher weight
        w = np.exp(logw - logw.max()); w = w / w.sum()
        # global RNG (seeded once per run) -> reproducible yet varies across iterations
        idx = np.random.choice(len(progs), size=k, replace=False, p=w)
        chosen = sorted([progs[i] for i in idx], key=lambda d: d["select_score"])
    return "\n\n".join(
        f"# select_score={d['select_score']:.2f}  cv_mse={d['cv_mse']:.2f}  "
        f"n_params={d['n_params']}  params={np.round(d['params'],3).tolist()}\n{d['code'].strip()}"
        for d in chosen)

def run_llm_sr(task, max_iterations=300, n_islands=10, patience=120,
               min_delta=0.01, temperature=0.9, max_usd=None, max_minutes=None, verbose=True):
    cfg = TASKS[task]
    Xtr, ytr, Xte, yte, bte = get_arrays(task)
    np.random.seed(GLOBAL_SEED)                 # reproducible island bookkeeping
    t0 = time.time()
    print(f"\n{'='*78}\nROBUST-SELECTION LLM-SR -- task={task} ({cfg['fmt']})  backbone={MODEL}")
    print(f"max_iters={max_iterations}  islands={n_islands}  patience={patience}  "
          f"FIT_RESTARTS={FIT_RESTARTS}  N_FOLDS={N_FOLDS}  MAX_PARAMS={MAX_PARAMS}")
    print(f"SCP benchmark (test, observation only): pooled {cfg['scp']['pooled']:.2f}\n{'='*78}")

    seed = evaluate(cfg["seed"], Xtr, ytr, Xte, yte, bte)
    islands = [[seed] for _ in range(n_islands)]
    best = seed; top5 = [seed] if seed else []; history = []
    best_by_test = seed   # observation-only: lowest TEST ever seen (NOT used for selection)
    since = 0; consecutive_fail = 0
    if best:
        print(f"SEED  select={best['select_score']:.2f}  train={best['train_mse']:.2f}  "
              f"cv={best['cv_mse']:.2f}  test={best['test_mse']:.2f}")

    reset_every = 60  # periodically reset worst-performing islands to escape stagnation (paper)
    for it in range(1, max_iterations+1):
        if it % reset_every == 0:
            scored = []
            for k in range(n_islands):
                vals = [c['select_score'] for c in islands[k] if c]
                scored.append((min(vals) if vals else np.inf, k))
            scored.sort()
            survivors = [k for _, k in scored[:max(1, n_islands//2)]]
            losers    = [k for _, k in scored[max(1, n_islands//2):]]
            for k in losers:
                src = islands[np.random.choice(survivors)]
                seedbest = min([c for c in src if c], key=lambda d: d['select_score'])
                islands[k] = [seedbest]
        isl = it % n_islands
        try:
            raw = call_llm(build_prompt(task, fmt_exp(islands[isl])), temperature=temperature, nonce=it)
            cand = evaluate(extract_code(raw), Xtr, ytr, Xte, yte, bte)
        except Exception as e:
            cand = None; consecutive_fail += 1
            if consecutive_fail <= 3 or consecutive_fail % 25 == 0:
                print(f"    [call/eval error #{consecutive_fail}] {type(e).__name__}: {str(e)[:200]}", flush=True)
            if consecutive_fail >= 20:
                print(f"[ABORT] {consecutive_fail} consecutive failures (last: {type(e).__name__})."); break

        if cand is not None:
            consecutive_fail = 0
            # OBSERVATION-ONLY: best generalisation ever seen (does NOT affect the search)
            if best_by_test is None or cand["test_mse"] < best_by_test["test_mse"]:
                best_by_test = cand
                print(f"    >>> new best TEST {cand['test_mse']:.2f} at iter {it} "
                      f"(select {cand['select_score']:.2f})  [observation-only]", flush=True)
            isl_best = min([c["select_score"] for c in islands[isl]]) if islands[isl] else np.inf
            if cand["select_score"] < isl_best:
                islands[isl].append(cand)
            if best is None or cand["select_score"] < best["select_score"] - min_delta:
                best = cand; since = 0
                top5 = sorted([c for c in top5+[cand] if c], key=lambda d: d["select_score"])[:5]
            else:
                since += 1
            history.append((it, cand["train_mse"], cand["test_mse"],
                            best["train_mse"], best["test_mse"], best["select_score"]))
        else:
            since += 1

        if verbose:
            cand_sel = cand['select_score'] if cand is not None else float('nan')
            print(f"  iter {it:4d} | cand select {cand_sel:8.2f} | best select {best['select_score']:7.2f} | "
                  f"best train {best['train_mse']:7.2f} | best test {best['test_mse']:7.2f} "
                  f"(SCP {cfg['scp']['pooled']:.1f}) | {since}/{patience} stale | ${SPEND['usd']:.2f}", flush=True)

        if since >= patience:
            print(f"[CONVERGED] no select-score improvement > {min_delta} for {patience} iters (iter {it})."); break
        if max_usd is not None and SPEND["usd"] >= max_usd:
            print(f"[BUDGET STOP] reached ${max_usd:.2f} at iter {it}."); break
        if max_minutes is not None and (time.time() - t0) >= max_minutes * 60:
            print(f"[TIME STOP] reached {max_minutes} min at iter {it}."); break

    return dict(task=task, best=best, best_by_test=best_by_test, top5=top5, history=history,
                best_fn=compile_equation(best["code"]) if best else None,
                best_params=best["params"] if best else None)
print("Search loop ready (selection by robustness score; best_by_test stays observation-only).")


Search loop ready (selection by robustness score; best_by_test stays observation-only).


## 8. Run RA (start here — the hard case)
`max_usd` is a hard safety stop. With the cache, re-running this cell after a kernel restart costs nothing for already-seen prompts.

In [26]:
# Pilot first: set max_iterations=30, max_usd=5.0 to gauge Opus cost/iteration, then scale up.
# Opus is ~5x the price of Sonnet, so watch the live $ counter.
MAX_MINUTES_RA = 180
SPEND["usd"] = 0
ra_results = run_llm_sr("RA", max_iterations=400, n_islands=10, patience=120,
                        min_delta=0.01, temperature=0.9, max_usd=10.0,
                        max_minutes=MAX_MINUTES_RA, verbose=True)



ROBUST-SELECTION LLM-SR -- task=RA (ex-ante reserve-price)  backbone=claude-sonnet-4-6
max_iters=400  islands=10  patience=120  FIT_RESTARTS=20  N_FOLDS=5  MAX_PARAMS=4
SCP benchmark (test, observation only): pooled 73.50
SEED  select=1271.77  train=269.71  cv=269.77  test=692.77
    >>> new best TEST 69.65 at iter 1 (select 278.55)  [observation-only]
  iter    1 | cand select   278.55 | best select  278.55 | best train  257.05 | best test   69.65 (SCP 73.5) | 0/120 stale | $0.02
  iter    2 | cand select   278.55 | best select  278.55 | best train  257.05 | best test   69.65 (SCP 73.5) | 1/120 stale | $0.05
  iter    3 | cand select   278.35 | best select  278.35 | best train  256.68 | best test   69.77 (SCP 73.5) | 0/120 stale | $0.07
    >>> new best TEST 68.20 at iter 4 (select 284.88)  [observation-only]
  iter    4 | cand select   284.88 | best select  278.35 | best train  256.68 | best test   69.77 (SCP 73.5) | 1/120 stale | $0.10
    >>> new best TEST 68.16 at iter 5 (select 

## 9. Run AB

In [27]:
MAX_MINUTES_AB = 180
SPEND["usd"] = 0
ab_results = run_llm_sr("AB", max_iterations=400, n_islands=10, patience=120,
                        min_delta=0.01, temperature=0.9, max_usd=10.0,
                        max_minutes=MAX_MINUTES_AB, verbose=True)


ROBUST-SELECTION LLM-SR -- task=AB (ex-post counter-offer)  backbone=claude-sonnet-4-6
max_iters=400  islands=10  patience=120  FIT_RESTARTS=20  N_FOLDS=5  MAX_PARAMS=4
SCP benchmark (test, observation only): pooled 64.10
SEED  select=1278.05  train=276.04  cv=276.05  test=507.26
    >>> new best TEST 67.85 at iter 1 (select 152.45)  [observation-only]
  iter    1 | cand select   152.45 | best select  152.45 | best train  139.97 | best test   67.85 (SCP 64.1) | 0/120 stale | $0.02
  iter    2 | cand select      nan | best select  152.45 | best train  139.97 | best test   67.85 (SCP 64.1) | 1/120 stale | $0.05
    >>> new best TEST 60.77 at iter 3 (select 149.45)  [observation-only]
  iter    3 | cand select   149.45 | best select  149.45 | best train  136.61 | best test   60.77 (SCP 64.1) | 0/120 stale | $0.07
  iter    4 | cand select   147.89 | best select  147.89 | best train  137.37 | best test   67.14 (SCP 64.1) | 0/120 stale | $0.09
  iter    5 | cand select   152.37 | best sele

# Assessment

In [28]:
# ============================================================================
# FINALIST ASSESSMENT — diagnostics (TEST-BLIND).
# ============================================================================
SELECT_TIE_BAND = 3.0        # finalists within this many select-score points of the best are "tied"
CV_TIE          = 5.0        # ...AND within this many CV-MSE points of the best CV (keeps ties genuine)
PROBE_SWEEP     = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]   # synthetic cost shapes on TRAIN inputs (test power=4 EXCLUDED)
PARAM_EXTREME   = 8.0        # |raw param| above this flags a blown-up / boundary-pinned fit

def _cv_mean_std(fn, X, y, n_params):
    """Re-run k-fold CV; return (mean, std) of held-out MSE. std = fold-to-fold stability."""
    N = len(y); h = make_helpers(POWER_TRAIN)
    idx = np.random.default_rng(GLOBAL_SEED + 77).permutation(N)
    folds = np.array_split(idx, N_FOLDS); errs = []
    for k in range(N_FOLDS):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(N_FOLDS) if j != k])
        p, _ = fit_params(fn, X[tr], y[tr], POWER_TRAIN, n_params, restarts=4, seed0=k+1)
        if p is None: return (np.nan, np.nan)
        try: pred = np.asarray(fn(X[te], p, h), float)
        except Exception: return (np.nan, np.nan)
        if not np.all(np.isfinite(pred)): return (np.nan, np.nan)
        errs.append(mse(pred, y[te]))
    return (float(np.mean(errs)), float(np.std(errs)))

def _probe_smoothness(fn, X, params):
    """Mean prediction across synthetic cost shapes. Returns (all_finite, max_relative_jump)."""
    ms = []
    for pw in PROBE_SWEEP:
        try: pp = np.asarray(fn(X, params, make_helpers(pw)), float)
        except Exception: return (False, np.inf)
        if not np.all(np.isfinite(pp)): return (False, np.inf)
        ms.append(float(np.mean(pp)))
    ms = np.array(ms); scale = abs(np.mean(ms)) + 1e-9
    jumps = np.abs(np.diff(ms)) / scale
    return (True, float(np.max(jumps)) if len(jumps) else 0.0)

def _monotonic_in_value(fn, X, params, vcol=0, bump=5.0):
    """Fraction of rows where the prediction RISES when value rises (reserve/offer should)."""
    h = make_helpers(POWER_TRAIN)
    try:
        base = np.asarray(fn(X, params, h), float)
        Xb = X.copy().astype(float); Xb[:, vcol] = Xb[:, vcol] + bump
        up = np.asarray(fn(Xb, params, h), float)
    except Exception: return np.nan
    m = np.isfinite(base) & np.isfinite(up)
    if not m.any(): return np.nan
    return float(np.mean(up[m] >= base[m] - 1e-9))

def _param_health(params):
    if params is None or len(np.atleast_1d(params)) == 0: return 0.0
    return float(np.max(np.abs(np.asarray(params, float))))

def _preds_under(band, X, power):
    """(M, N) matrix: each band member's predictions on X under the cost distribution `power`."""
    h = make_helpers(power); out = []
    for r in band:
        fn = compile_equation(r["code"])
        try: p = np.asarray(fn(X, r["params"], h), float)
        except Exception: p = np.full(X.shape[0], np.nan)
        out.append(p)
    return np.array(out)

def _consensus_disagreement(band, X):
    """How far each tied equation strays from the MAJORITY, measured on the uniform train AND the
       probe distributions (test power=4 NEVER used). On uniform they agree (that is why they tie);
       an over-fit equation reveals itself by DIVERGING as the distribution shifts. Lower = consensus."""
    powers = [POWER_TRAIN] + PROBE_SWEEP
    M = len(band); dis = np.zeros(M); cnt = 0
    for pw in powers:
        P = _preds_under(band, X, pw)                       # (M, N)
        keep = np.all(np.isfinite(P), axis=0)               # columns finite for every member
        if keep.sum() == 0: continue
        Pk = P[:, keep]; cons = np.median(Pk, axis=0)       # majority prediction
        dis += np.mean(np.abs(Pk - cons), axis=1); cnt += 1
    return (dis / max(cnt, 1)) if cnt else np.zeros(M)

def assess_finalists(results, k=5, tie_band=SELECT_TIE_BAND, cv_tie=CV_TIE):
    """Test-blind re-ranking of the top-K finalists. Prints a diagnostics table, a robust
       band ENSEMBLE, and returns the consensus pick (original finalist dict)."""
    task = results["task"]; cfg = TASKS[task]
    Xtr, ytr, Xte, yte, bte = get_arrays(task)
    vcol = cfg["pred_cols"].index("value") if "value" in cfg["pred_cols"] else 0
    pool = sorted([c for c in results.get("top5", []) if c], key=lambda d: d["select_score"])[:k]
    if not pool:
        print("no finalists in results['top5']"); return None
    best_sel = pool[0]["select_score"]; rows = []
    for c in pool:
        fn = compile_equation(c["code"]); p = c["params"]
        cvm, cvs = _cv_mean_std(fn, Xtr, ytr, c["n_params"])
        ok, jump = _probe_smoothness(fn, Xtr, p)
        mono = _monotonic_in_value(fn, Xtr, p, vcol)
        rows.append(dict(orig=c, params=c["params"], sel=c["select_score"], dsel=c["select_score"] - best_sel,
                         cv=cvm, cv_sd=cvs, probe_ok=ok, jump=jump, mono=mono, disag=np.nan,
                         pmax=_param_health(p), npar=c["n_params"], clen=len(c["code"].strip()),
                         test=c["test_mse"], code=c["code"].strip()))
    finite_cv = [r["cv"] for r in rows if not (isinstance(r["cv"], float) and np.isnan(r["cv"]))]
    best_cv = min(finite_cv) if finite_cv else np.inf
    band = [r for r in rows if r["dsel"] <= tie_band and r["probe_ok"]
            and not (isinstance(r["cv"], float) and np.isnan(r["cv"])) and r["cv"] <= best_cv + cv_tie]
    if not band: band = [r for r in rows if r["dsel"] <= tie_band] or rows

    # ---- consensus disagreement among the ties (TEST-BLIND) -> primary tie-breaker ----
    if len(band) >= 2:
        dis = _consensus_disagreement(band, Xtr)
        for r, d in zip(band, dis): r["disag"] = float(d)
    else:
        band[0]["disag"] = 0.0
    # rank: parsimony -> consensus (agrees with majority under shift) -> stability -> shorter
    band_sorted = sorted(band, key=lambda r: (r["npar"], r["disag"], r["cv_sd"], r["clen"]))
    pick = band_sorted[0]

    # ---- band ENSEMBLE: median prediction across all ties (dilutes any single outlier) ----
    Pte = _preds_under(band, Xte, POWER_TEST)               # observation-only (reporting)
    ens = np.median(Pte, axis=0)
    ens_test = mse_by_n(ens, yte, bte)["pooled"] if np.all(np.isfinite(ens)) else float("nan")
    band_median_test = float(np.median([r["test"] for r in band]))

    print("\n" + "=" * 100)
    print(f"FINALIST ASSESSMENT — task={task} ({cfg['fmt']})    SCP test = {cfg['scp']['pooled']:.2f}")
    print(f"ties = best + {tie_band:.1f} select AND + {cv_tie:.1f} CV   |   TEST-BLIND rank: parsimony > consensus > stability")
    print("=" * 100)
    hdr = (f"{'#':>2} {'select':>7} {'dsel':>5} {'cv':>6} {'cv_sd':>6} {'disag':>6} {'jump':>6} "
           f"{'mono':>5} {'|p|max':>6} {'npar':>4} {'len':>5} | {'TEST*':>7}")
    print(hdr); print("-" * len(hdr))
    for i, r in enumerate(rows):
        tag = "  <== PICK" if r is pick else ("  (tie)" if r in band else "")
        mono = ("%.2f" % r["mono"]) if not (isinstance(r["mono"], float) and np.isnan(r["mono"])) else "  na"
        disg = ("%.2f" % r["disag"]) if not (isinstance(r["disag"], float) and np.isnan(r["disag"])) else "   na"
        flag = "!" if r["pmax"] > PARAM_EXTREME else " "
        print(f"{i:>2} {r['sel']:>7.2f} {r['dsel']:>5.2f} {r['cv']:>6.1f} {r['cv_sd']:>6.1f} {disg:>6} "
              f"{r['jump']:>6.2f} {mono:>5} {r['pmax']:>5.2f}{flag} {r['npar']:>4} {r['clen']:>5} | {r['test']:>7.2f}{tag}")
    print("-" * len(hdr))
    print("* TEST = out-of-sample (power=4), REPORTING ONLY — never used to choose.   '!' = |param| extreme.\n")
    reach = results.get("best_by_test")
    print(f"CONSENSUS PICK (test-blind) : select={pick['sel']:.2f}  disag={pick['disag']:.2f}  npar={pick['npar']}  ->  test={pick['test']:.2f}  (SCP {cfg['scp']['pooled']:.2f})")
    print(f"BAND ENSEMBLE (median of {len(band)} ties, robust headline)        ->  test={ens_test:.2f}")
    print(f"band median test = {band_median_test:.2f}   |   argmin-select pointer test = {rows[0]['test']:.2f}")
    if reach is not None:
        print(f"REACH ceiling (oracle, NOT a result): best test anywhere = {reach['test_mse']:.2f}")
    print("\nCONSENSUS-PICK EQUATION:\n" + pick["code"])
    return pick["orig"]

print("Finalist-assessment functions ready (consensus + ensemble). Call assess_finalists(ra_results) / assess_finalists(ab_results).")


Finalist-assessment functions ready (consensus + ensemble). Call assess_finalists(ra_results) / assess_finalists(ab_results).


In [29]:
# Run the assessment on whichever runs have finished.
# k = how many finalists to inspect; tie_band = select-score window treated as a tie.
if "ra_results" in dir(): assess_finalists(ra_results, k=5, tie_band=3.0)
if "ab_results" in dir(): assess_finalists(ab_results, k=5, tie_band=3.0)



FINALIST ASSESSMENT — task=RA (ex-ante reserve-price)    SCP test = 73.50
ties = best + 3.0 select AND + 5.0 CV   |   TEST-BLIND rank: parsimony > consensus > stability
 #  select  dsel     cv  cv_sd  disag   jump  mono |p|max npar   len |   TEST*
------------------------------------------------------------------------------
 0  261.82  0.00  253.8    8.1   0.00   0.13  1.00  1.00     4  1853 |   73.73  <== PICK
 1  276.18 14.35  268.2   11.5     na   0.14  0.92  1.00     4  1910 |   69.86
 2  277.72 15.89  269.7   11.6     na   0.14  1.00  1.00     4  1289 |   69.74
 3  278.35 16.53  270.4   11.4     na   0.14  1.00  1.00     4  2269 |   69.77
 4  278.55 16.73  270.6   11.5     na   0.14  1.00  1.00     4  2081 |   69.65
------------------------------------------------------------------------------
* TEST = out-of-sample (power=4), REPORTING ONLY — never used to choose.   '!' = |param| extreme.

CONSENSUS PICK (test-blind) : select=261.82  disag=0.00  npar=4  ->  test=73.73  (SCP 73.

## MSE breakdown for the selected equation

Train (uniform, power=1) and test (left-skewed, power=4) MSE for one chosen equation, split by number of bidders n=1,2,3 and pooled, with SCP's test MSE alongside. Pass the equation returned by `assess_finalists` (the consensus pick), or any finalist from `results["top5"]`.

In [30]:
# ============================================================================
# MSE BREAKDOWN — train & test MSE by number of bidders (n=1,2,3) and pooled,
# for a chosen equation (e.g. the consensus pick returned by assess_finalists).
# TRAIN = uniform (power=1); TEST = left-skewed (power=4). Reuses the notebook engine.
# ============================================================================
def report_mse(results, finalist, label=None):
    task = results["task"]; cfg = TASKS[task]
    Xtr, ytr, Xte, yte, bte = get_arrays(task)
    bcol = cfg["pred_cols"].index("bidN")
    btr = Xtr[:, bcol].astype(int)
    fn = compile_equation(finalist["code"]); p = finalist["params"]
    ptr = np.asarray(fn(Xtr, p, make_helpers(POWER_TRAIN)), float)   # uniform train
    pte = np.asarray(fn(Xte, p, make_helpers(POWER_TEST)),  float)   # left-skewed test
    tr = mse_by_n(ptr, ytr, btr)
    te = mse_by_n(pte, yte, bte)
    scp = cfg.get("scp", {})
    def scp_get(key): return scp.get(key, scp.get(str(key), float("nan")))
    name = label or f"selected (select={finalist.get('select_score', float('nan')):.2f}, npar={finalist.get('n_params','?')})"
    print("\n" + "=" * 74)
    print(f"MSE BREAKDOWN — task={task} ({cfg['fmt']})")
    print(f"equation: {name}")
    print("=" * 74)
    print(f"{'n':>7} {'N_train':>8} {'N_test':>7} | {'TRAIN_MSE':>10} {'TEST_MSE':>10} | {'SCP_TEST':>9}")
    print("-" * 74)
    for n in [1, 2, 3]:
        ntr = int(np.sum(btr == n)); nte = int(np.sum(bte == n))
        print(f"{n:>7} {ntr:>8} {nte:>7} | {tr.get(n, float('nan')):>10.2f} "
              f"{te.get(n, float('nan')):>10.2f} | {scp_get(n):>9.2f}")
    print("-" * 74)
    print(f"{'pooled':>7} {len(ytr):>8} {len(yte):>7} | {tr['pooled']:>10.2f} "
          f"{te['pooled']:>10.2f} | {scp_get('pooled'):>9.2f}")
    print("=" * 74)
    return {"train": tr, "test": te, "scp": {k: scp_get(k) for k in [1, 2, 3, 'pooled']}}

print("report_mse ready. Usage:  pick = assess_finalists(ra_results);  report_mse(ra_results, pick)")


report_mse ready. Usage:  pick = assess_finalists(ra_results);  report_mse(ra_results, pick)


In [31]:
# Train/test MSE (by n and pooled) for the SELECTED equation of each finished task.
# assess_finalists() prints its finalist table and returns the consensus pick;
# report_mse() then breaks that pick down by number of bidders.
if "ra_results" in dir():
    pick_ra = assess_finalists(ra_results)
    report_mse(ra_results, pick_ra)
if "ab_results" in dir():
    pick_ab = assess_finalists(ab_results)
    report_mse(ab_results, pick_ab)



FINALIST ASSESSMENT — task=RA (ex-ante reserve-price)    SCP test = 73.50
ties = best + 3.0 select AND + 5.0 CV   |   TEST-BLIND rank: parsimony > consensus > stability
 #  select  dsel     cv  cv_sd  disag   jump  mono |p|max npar   len |   TEST*
------------------------------------------------------------------------------
 0  261.82  0.00  253.8    8.1   0.00   0.13  1.00  1.00     4  1853 |   73.73  <== PICK
 1  276.18 14.35  268.2   11.5     na   0.14  0.92  1.00     4  1910 |   69.86
 2  277.72 15.89  269.7   11.6     na   0.14  1.00  1.00     4  1289 |   69.74
 3  278.35 16.53  270.4   11.4     na   0.14  1.00  1.00     4  2269 |   69.77
 4  278.55 16.73  270.6   11.5     na   0.14  1.00  1.00     4  2081 |   69.65
------------------------------------------------------------------------------
* TEST = out-of-sample (power=4), REPORTING ONLY — never used to choose.   '!' = |param| extreme.

CONSENSUS PICK (test-blind) : select=261.82  disag=0.00  npar=4  ->  test=73.73  (SCP 73.

## Significance vs SCP (paper) for the selected equation

Compares the selected (consensus-pick) equation to SCP on the left-skewed test set, using the same method as the scenario notebooks: cluster squared errors by auctioneer (`name`, 64 auctioneers), then a paired t-test and Wilcoxon signed-rank on the per-auctioneer MSE differences. SCP here is the recomputed benchmark using the paper's published parameters (μ,λ), so its pooled MSE matches the paper. A negative mean difference means the discovered equation beats SCP. Relies on the scenario notebook having run cell 9 (the `_scp` column) and cell 5 (the `name` column).

In [32]:
# ============================================================================
# SIGNIFICANCE vs SCP (paper params) for a chosen equation — per-auctioneer
# paired test, identical method to the scenario notebooks' cell 31.
# Needs the scenario notebook's test frame with columns "_scp" (cell 9) and "name" (cell 5).
# ============================================================================
import numpy as np, pandas as pd
from scipy import stats

def _bounded_pred_eq(task, dte, fn, params):
    pred = np.asarray(fn(dte[TASKS[task]["pred_cols"]].values, params, make_helpers(POWER_TEST)), float)
    if task == "RA":
        return np.clip(pred, 0, 100)
    return np.minimum(np.clip(pred, 0, None), dte["auctionprice"].values)   # AB: [0, auctionprice]

def _per_name_mse(pred, actual, names):
    df = pd.DataFrame({"se": (np.asarray(pred) - np.asarray(actual))**2, "name": np.asarray(names)})
    return df.groupby("name")["se"].mean()

def significance_vs_scp_finalist(results, finalist, eq_label="selected (consensus pick)"):
    task = results["task"]; cfg = TASKS[task]; dte = globals()[cfg["test"]]
    fn = compile_equation(finalist["code"])
    pred   = _bounded_pred_eq(task, dte, fn, finalist["params"])
    actual = dte[cfg["target"]].values
    scp    = dte["_scp"].values
    names  = dte["name"].values

    m_model = _per_name_mse(pred, actual, names)
    m_scp   = _per_name_mse(scp,  actual, names)
    common  = m_model.index.intersection(m_scp.index)
    a, s    = m_model.loc[common].values, m_scp.loc[common].values
    diff    = a - s                                       # model - SCP; negative => model better

    t_stat, t_p = stats.ttest_rel(a, s)
    try:    w_stat, w_p = stats.wilcoxon(a, s)
    except ValueError: w_stat, w_p = float("nan"), float("nan")

    pooled_model = float(np.mean((pred - actual)**2))
    pooled_scp   = float(np.mean((scp  - actual)**2))

    print("=" * 82)
    print(f"{task}: {eq_label} vs SCP (paper params), left-skewed TEST set  ({len(common)} auctioneers)")
    print("=" * 82)
    rel = "BEATS" if pooled_model < pooled_scp else "does NOT beat"
    print(f"pooled MSE : model {pooled_model:7.2f}   |   SCP {pooled_scp:7.2f}   ({rel} SCP by {abs(pooled_model-pooled_scp):.2f})")
    print(f"per-auctioneer mean MSE diff (model - SCP): {diff.mean():+.3f}   (negative = model better)")
    print(f"   median diff {np.median(diff):+.3f}   |   auctioneers with model < SCP: {int(np.sum(diff<0))}/{len(common)}")
    print(f"paired t-test        : t = {t_stat:+.3f},  p = {t_p:.4f}")
    print(f"Wilcoxon signed-rank : W = {w_stat:.1f},  p = {w_p:.4f}")
    direction = "better than" if diff.mean() < 0 else "worse than"
    sig = "significant" if (np.isfinite(t_p) and t_p < 0.05) else "not significant (n.s.)"
    print(f"=> on average {direction} SCP; difference from zero is {sig} at alpha=0.05.")
    print("=" * 82)
    return dict(task=task, eq_label=eq_label, pooled_model=pooled_model, pooled_scp=pooled_scp,
                mean_diff=float(diff.mean()), median_diff=float(np.median(diff)),
                t_stat=float(t_stat), t_p=float(t_p), w_stat=float(w_stat), w_p=float(w_p),
                n_clusters=int(len(common)), n_model_better=int(np.sum(diff < 0)))

print("significance_vs_scp_finalist ready.  Usage: significance_vs_scp_finalist(ra_results, pick_ra)")


significance_vs_scp_finalist ready.  Usage: significance_vs_scp_finalist(ra_results, pick_ra)


In [33]:
# Significance of the SELECTED (consensus-pick) equation vs SCP (paper), per-auctioneer paired test.
# Reuses pick_ra / pick_ab from the MSE-breakdown cell if present; otherwise computes them.
if "ra_results" in dir():
    pick_ra = pick_ra if "pick_ra" in dir() else assess_finalists(ra_results)
    significance_vs_scp_finalist(ra_results, pick_ra)
if "ab_results" in dir():
    pick_ab = pick_ab if "pick_ab" in dir() else assess_finalists(ab_results)
    significance_vs_scp_finalist(ab_results, pick_ab)


RA: selected (consensus pick) vs SCP (paper params), left-skewed TEST set  (64 auctioneers)
pooled MSE : model   73.73   |   SCP   73.50   (does NOT beat SCP by 0.23)
per-auctioneer mean MSE diff (model - SCP): +0.229   (negative = model better)
   median diff +1.626   |   auctioneers with model < SCP: 31/64
paired t-test        : t = +0.125,  p = 0.9013
Wilcoxon signed-rank : W = 1038.0,  p = 0.9893
=> on average worse than SCP; difference from zero is not significant (n.s.) at alpha=0.05.
AB: selected (consensus pick) vs SCP (paper params), left-skewed TEST set  (64 auctioneers)
pooled MSE : model   61.62   |   SCP   64.10   (BEATS SCP by 2.48)
per-auctioneer mean MSE diff (model - SCP): -1.897   (negative = model better)
   median diff -2.422   |   auctioneers with model < SCP: 39/64
paired t-test        : t = -2.188,  p = 0.0324
Wilcoxon signed-rank : W = 672.0,  p = 0.0139
=> on average better than SCP; difference from zero is significant at alpha=0.05.


In [34]:
# ============================================================================
# SIGNIFICANCE vs SCP by number of bidders (n=1,2,3) AND pooled, for one format.
# Per-auctioneer paired test (t-test + Wilcoxon), same clustering as the pooled version.
# Needs the test frame columns "_scp" (cell 9) and "name" (cell 5).
# ============================================================================
import numpy as np, pandas as pd
from scipy import stats

def _bounded_pred_eq(task, dte, fn, params):
    pred = np.asarray(fn(dte[TASKS[task]["pred_cols"]].values, params, make_helpers(POWER_TEST)), float)
    if task == "RA":
        return np.clip(pred, 0, 100)
    return np.minimum(np.clip(pred, 0, None), dte["auctionprice"].values)   # AB: [0, auctionprice]

def _per_name_mse(se, names):
    return pd.DataFrame({"se": se, "name": names}).groupby("name")["se"].mean()

def significance_by_n(results, finalist, eq_label="consensus pick"):
    task = results["task"]; cfg = TASKS[task]; dte = globals()[cfg["test"]]
    fn   = compile_equation(finalist["code"])
    pred   = _bounded_pred_eq(task, dte, fn, finalist["params"])
    actual = dte[cfg["target"]].values
    scp    = dte["_scp"].values
    names  = dte["name"].values
    bidN   = dte["bidN"].values.astype(int)

    se_model = (pred - actual) ** 2
    se_scp   = (scp  - actual) ** 2

    rows = []
    for grp in [1, 2, 3, "pooled"]:
        m = np.ones(len(actual), bool) if grp == "pooled" else (bidN == grp)
        mm = _per_name_mse(se_model[m], names[m])          # per-auctioneer mean MSE, model
        ms = _per_name_mse(se_scp[m],   names[m])          # per-auctioneer mean MSE, SCP
        common = mm.index.intersection(ms.index)
        a, s = mm.loc[common].values, ms.loc[common].values
        diff = a - s                                       # model - SCP; negative => model better
        t_stat, t_p = stats.ttest_rel(a, s)
        try:    w_stat, w_p = stats.wilcoxon(a, s)
        except ValueError: w_stat, w_p = np.nan, np.nan
        rows.append({"n": grp, "N_rows": int(m.sum()), "N_auct": int(len(common)),
                     "model_MSE": round(float(np.mean(se_model[m])), 2),
                     "SCP_MSE":   round(float(np.mean(se_scp[m])), 2),
                     "mean_diff": round(float(diff.mean()), 3),
                     "n_better":  int(np.sum(diff < 0)),
                     "t_p": round(float(t_p), 4), "w_p": round(float(w_p), 4),
                     "sig@0.05": "yes" if (np.isfinite(t_p) and t_p < 0.05) else "no"})
    df = pd.DataFrame(rows).set_index("n")
    print("=" * 96)
    print(f"{task} ({cfg['fmt']}): {eq_label} vs SCP, per-n and pooled  (paired per-auctioneer test)")
    print("mean_diff = mean per-auctioneer (model MSE - SCP MSE); negative = discovered equation better")
    print("=" * 96)
    display(df)
    return df

# ---- run for both formats using the consensus picks from assess_finalists() ----
if "ra_results" in dir():
    pick_ra = pick_ra if "pick_ra" in dir() else assess_finalists(ra_results)
    sig_ra = significance_by_n(ra_results, pick_ra)
if "ab_results" in dir():
    pick_ab = pick_ab if "pick_ab" in dir() else assess_finalists(ab_results)
    sig_ab = significance_by_n(ab_results, pick_ab)


RA (ex-ante reserve-price): consensus pick vs SCP, per-n and pooled  (paired per-auctioneer test)
mean_diff = mean per-auctioneer (model MSE - SCP MSE); negative = discovered equation better


,N_rows,N_auct,model_MSE,SCP_MSE,mean_diff,n_better,t_p,w_p,sig@0.05
n,,,,,,,,,
1,1920,64,53.05,49.51,3.536,33,0.0967,0.6445,no
2,1920,64,85.83,81.51,4.317,30,0.2448,0.3389,no
3,1920,64,82.31,89.48,-7.167,42,0.0056,0.0023,yes
pooled,5760,64,73.73,73.50,0.229,31,0.9013,0.9893,no


AB (ex-post counter-offer): consensus pick vs SCP, per-n and pooled  (paired per-auctioneer test)
mean_diff = mean per-auctioneer (model MSE - SCP MSE); negative = discovered equation better


,N_rows,N_auct,model_MSE,SCP_MSE,mean_diff,n_better,t_p,w_p,sig@0.05
n,,,,,,,,,
1,1637,64,76.86,78.24,-0.764,42,0.5628,0.2612,no
2,1480,64,59.06,62.52,-2.820,35,0.0204,0.0881,yes
3,1303,64,45.36,48.12,-1.850,37,0.0763,0.0541,no
pooled,4420,64,61.62,64.10,-1.897,39,0.0324,0.0139,yes
